# TrustGate × SWE-bench Verified — Machine Learning Study

## Goal

Build a defensible machine-learning study for **TrustGate**:

> Given a GitHub issue, repository state, and a candidate PR patch, estimate the probability that the patch correctly resolves the underlying issue.

The final TrustGate decision is a **three-way gate**:

- **Approve** — high-confidence candidate
- **Human review** — uncertain candidate
- **Reject** — low-confidence candidate

The model should produce a probability first. The three-way decision is then created from thresholds chosen on validation data.

### Important definition

A SWE-bench success label means:

> the candidate patch satisfies the benchmark's issue-linked tests in the benchmark evaluation environment.

That is a useful software-correctness proxy. It is **not** by itself a complete label for production PR safety, code quality, security, maintainability, or organizational approval policy.

### Notebook principles

1. Understand the benchmark before modeling it.
2. Keep benchmark-only information separate from real TrustGate features.
3. Generate candidate-patch labels with the official SWE-bench evaluation environment.
4. Split data by **issue**, not by candidate row.
5. Report discrimination, classification, calibration, and decision metrics.
6. Treat evaluation/environment failures separately from genuine patch failures.
7. Use synthetic candidates only for pipeline smoke tests, not as the main evidence of ML performance.

## 1. SWE-bench Verified in simple English

Imagine 500 real bug reports taken from GitHub.

For each one, SWE-bench tells us:

- **what the user/developer reported** — `problem_statement`
- **which repository** the problem came from — `repo`
- **the exact code state before the fix** — `base_commit`
- **the real patch that fixed it** — `patch`
- **the tests connected to the fix** — `test_patch`
- **which tests are expected to change from failing to passing** — `FAIL_TO_PASS`
- **which tests should continue passing** — `PASS_TO_PASS`
- **how to prepare the repository environment** — `environment_setup_commit`, `version`
- some historical context such as `hints_text` and `created_at`

SWE-bench Verified contains **500 human-validated tasks across 12 Python repositories**.

The key TrustGate transformation is:

```text
GitHub issue + codebase + candidate PR
                  |
                  v
        TrustGate features
                  |
                  v
      probability candidate is correct
                  |
        +---------+---------+
        |         |         |
     APPROVE   HUMAN      REJECT
               REVIEW
```

SWE-bench is valuable because the candidate patch can be executed in a controlled benchmark environment and checked against issue-linked tests.

**Primary source:** SWE-bench official dataset:
https://huggingface.co/datasets/SWE-bench/SWE-bench_Verified

**Princeton-hosted copy:**
https://huggingface.co/datasets/princeton-nlp/SWE-bench_Verified

**Source:** SWE-bench evaluation documentation:
https://github.com/SWE-bench/SWE-bench/blob/main/docs/assets/evaluation.md

## 2. What exactly is in the 500-row dataset?

The current Hugging Face dataset card reports one `test` split with **500 rows** and these fields:

| Field | Plain-English meaning | TrustGate use |
|---|---|---|
| `repo` | GitHub repository | EDA; optional production context |
| `instance_id` | Unique benchmark issue identifier | Grouping/splitting only |
| `base_commit` | Code version before the fix | Benchmark setup |
| `patch` | Reference/gold fix | Label reference only |
| `test_patch` | Tests added/changed by the real fix | Evaluation reference only |
| `problem_statement` | Issue title + issue body | Available to TrustGate |
| `hints_text` | Comments before the solution PR | Historical/context feature only if available |
| `created_at` | PR creation date | Temporal analysis |
| `version` | Evaluation/install version | Environment metadata |
| `FAIL_TO_PASS` | Tests expected to be fixed | Evaluation only; **not a production feature** |
| `PASS_TO_PASS` | Tests expected to stay green | Evaluation only; **not a production feature** |
| `environment_setup_commit` | Commit used to build/install the environment | Evaluation setup |
| `difficulty` | Benchmark difficulty label | EDA only by default |

The dataset card also reports that `repo` has 12 values and that the benchmark has 4 difficulty values.

### Why this matters

A common mistake is to train directly from these 500 rows.

That is too close to the benchmark-task level.

TrustGate makes decisions about **candidate PRs**, so the modeling table should look more like:

```text
issue A + candidate patch 1 -> label 1
issue A + candidate patch 2 -> label 0
issue A + candidate patch 3 -> label 1
issue B + candidate patch 4 -> label 0
...
```

One issue can therefore produce many candidate rows.

**Dataset source:** https://huggingface.co/datasets/princeton-nlp/SWE-bench_Verified

In [ ]:
# Analysis dependencies.
# This notebook is designed for Colab/Jupyter.
!pip -q install datasets scikit-learn pandas numpy matplotlib

In [ ]:
import ast
import json
import math
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

DATASET_NAME = "SWE-bench/SWE-bench_Verified"

try:
    ds = load_dataset(DATASET_NAME, split="test")
except Exception as official_error:
    # Compatibility with the original Princeton-hosted dataset card.
    print("Official SWE-bench dataset load failed:", official_error)
    DATASET_NAME = "princeton-nlp/SWE-bench_Verified"
    ds = load_dataset(DATASET_NAME, split="test")
swe = ds.to_pandas()

print("Rows:", len(swe))
print("Columns:", swe.columns.tolist())
display(swe.head(3))

In [ ]:
EXPECTED_COLUMNS = [
    "repo", "instance_id", "base_commit", "patch", "test_patch",
    "problem_statement", "hints_text", "created_at", "version",
    "FAIL_TO_PASS", "PASS_TO_PASS", "environment_setup_commit", "difficulty"
]

missing = [c for c in EXPECTED_COLUMNS if c not in swe.columns]
assert not missing, f"Missing expected columns: {missing}"

print("Unique repositories:", swe["repo"].nunique())
print("Difficulty values:", swe["difficulty"].value_counts(dropna=False).to_dict())
print(
    "Fraction where environment_setup_commit != base_commit:",
    (swe["environment_setup_commit"] != swe["base_commit"]).mean()
)
print("Unique versions:", swe["version"].nunique())
print("Rows with hints:", swe["hints_text"].fillna("").str.len().gt(0).mean())

## 3. Benchmark EDA

The first EDA section answers a simple question:

> What does this benchmark actually look like before we ask a model to learn anything?

We look at:

- repository balance
- difficulty balance
- patch size
- issue description size
- test counts
- test-patch size
- relationship between benchmark variables
- missing values
- environment differences

These plots describe the benchmark. They do **not** automatically define the TrustGate feature set.

In [ ]:
def parse_list_field(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return []
        try:
            value = json.loads(x)
            return value if isinstance(value, list) else [value]
        except Exception:
            # Keep the raw value as one item if it is not valid JSON.
            return [x]
    return []

def unified_diff_stats(patch_text):
    patch_text = patch_text or ""
    lines = patch_text.splitlines()

    additions = sum(
        1 for line in lines
        if line.startswith("+") and not line.startswith("+++")
    )
    deletions = sum(
        1 for line in lines
        if line.startswith("-") and not line.startswith("---")
    )
    files = re.findall(r"^\+\+\+ b/(.+)$", patch_text, flags=re.MULTILINE)
    hunks = len(re.findall(r"^@@", patch_text, flags=re.MULTILINE))

    return {
        "patch_additions": additions,
        "patch_deletions": deletions,
        "patch_files": len(set(files)),
        "patch_hunks": hunks,
        "patch_chars": len(patch_text),
    }

patch_stats = pd.DataFrame(
    [unified_diff_stats(p) for p in swe["patch"].fillna("")]
)

eda = pd.concat(
    [
        swe[
            [
                "instance_id", "repo", "difficulty", "version",
                "created_at", "environment_setup_commit", "base_commit"
            ]
        ].reset_index(drop=True),
        patch_stats.reset_index(drop=True),
    ],
    axis=1,
)

eda["problem_chars"] = swe["problem_statement"].fillna("").str.len().values
eda["problem_lines"] = swe["problem_statement"].fillna("").str.count(r"\n").values + 1
eda["hints_chars"] = swe["hints_text"].fillna("").str.len().values
eda["test_patch_chars"] = swe["test_patch"].fillna("").str.len().values
eda["fail_to_pass_count"] = swe["FAIL_TO_PASS"].map(lambda x: len(parse_list_field(x)))
eda["pass_to_pass_count"] = swe["PASS_TO_PASS"].map(lambda x: len(parse_list_field(x)))
eda["environment_diff"] = (
    swe["environment_setup_commit"] != swe["base_commit"]
).astype(int)

numeric_eda_cols = [
    "patch_additions", "patch_deletions", "patch_files", "patch_hunks",
    "patch_chars", "problem_chars", "problem_lines", "hints_chars",
    "test_patch_chars", "fail_to_pass_count", "pass_to_pass_count",
    "environment_diff"
]

display(eda[numeric_eda_cols].describe().T)

In [ ]:
# 3.1 Repository distribution
counts = swe["repo"].value_counts().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
plt.bar(counts.index, counts.values)
plt.title("SWE-bench Verified: tasks by repository")
plt.ylabel("Number of tasks")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# 3.2 Difficulty distribution
counts = swe["difficulty"].value_counts()

plt.figure(figsize=(8, 4))
plt.bar(counts.index.astype(str), counts.values)
plt.title("SWE-bench Verified: difficulty distribution")
plt.ylabel("Number of tasks")
plt.xlabel("Difficulty")
plt.tight_layout()
plt.show()

In [ ]:
# 3.3 Distribution of patch size
features_to_plot = [
    ("patch_additions", "Patch additions"),
    ("patch_deletions", "Patch deletions"),
    ("patch_files", "Files touched"),
    ("patch_hunks", "Patch hunks"),
]

for col, title in features_to_plot:
    clipped = eda[col].clip(upper=eda[col].quantile(0.99))

    plt.figure(figsize=(7, 4))
    plt.hist(clipped, bins=30)
    plt.title(title)
    plt.xlabel(col)
    plt.ylabel("Tasks")
    plt.tight_layout()
    plt.show()

In [ ]:
# 3.4 Issue and test complexity distributions
features_to_plot = [
    ("problem_chars", "Issue description length"),
    ("test_patch_chars", "Test patch size"),
    ("fail_to_pass_count", "FAIL_TO_PASS test count"),
    ("pass_to_pass_count", "PASS_TO_PASS test count"),
]

for col, title in features_to_plot:
    clipped = eda[col].clip(upper=eda[col].quantile(0.99))

    plt.figure(figsize=(7, 4))
    plt.hist(clipped, bins=30)
    plt.title(title)
    plt.xlabel(col)
    plt.ylabel("Tasks")
    plt.tight_layout()
    plt.show()

In [ ]:
# 3.5 Patch size by difficulty
plot_df = eda.copy()

order = [x for x in ["easy", "medium", "hard", "very hard"] if x in plot_df["difficulty"].astype(str).unique()]
if not order:
    order = sorted(plot_df["difficulty"].astype(str).unique())

groups = [
    plot_df.loc[plot_df["difficulty"].astype(str) == d, "patch_additions"].values
    for d in order
]

plt.figure(figsize=(9, 5))
plt.boxplot(groups, tick_labels=order, showfliers=False)
plt.title("Patch additions by benchmark difficulty")
plt.xlabel("Difficulty")
plt.ylabel("Added lines")
plt.tight_layout()
plt.show()

## 4. Heatmap: understand relationships between benchmark variables

The heatmap is useful because several measurements are naturally related.

For example:

- `patch_chars` will usually increase as additions/deletions increase.
- `test_patch_chars` can rise with the size of the change.
- issue length and patch complexity may or may not be related.
- test counts tell us about the verification footprint of the task.

A high correlation does **not** mean one variable causes another. It tells us that the two measurements move together.

In [ ]:
heat_cols = [
    "patch_additions", "patch_deletions", "patch_files", "patch_hunks",
    "patch_chars", "problem_chars", "test_patch_chars",
    "fail_to_pass_count", "pass_to_pass_count", "environment_diff"
]

corr = eda[heat_cols].corr()

plt.figure(figsize=(11, 8))
im = plt.imshow(corr.values, vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, label="Correlation")
plt.xticks(range(len(heat_cols)), heat_cols, rotation=60, ha="right")
plt.yticks(range(len(heat_cols)), heat_cols)
plt.title("SWE-bench Verified benchmark feature correlation heatmap")

for i in range(len(heat_cols)):
    for j in range(len(heat_cols)):
        plt.text(
            j, i, f"{corr.iloc[i, j]:.2f}",
            ha="center", va="center", fontsize=7
        )

plt.tight_layout()
plt.show()

In [ ]:
# 4.2 Missing-value heatmap
missing_rate = swe.isna().mean().sort_values(ascending=False)

plt.figure(figsize=(10, 4))
im = plt.imshow(
    missing_rate.values.reshape(1, -1),
    aspect="auto",
    cmap="viridis",
    vmin=0,
    vmax=max(0.01, float(missing_rate.max())),
)
plt.colorbar(im, label="Missing fraction")
plt.xticks(range(len(missing_rate)), missing_rate.index, rotation=70, ha="right")
plt.yticks([0], ["missing"])
plt.title("Missing-value profile across SWE-bench fields")
plt.tight_layout()
plt.show()

display(missing_rate.to_frame("missing_fraction"))

## 5. What should become a TrustGate feature?

This is the most important modeling idea in the notebook.

A **feature** is simply a measurable signal that the model can see when it is judging a candidate PR.

Think of a feature as one answer to one small question.

### A. Patch size and shape

- `patch_additions` — how many lines were added?
- `patch_deletions` — how many lines were removed?
- `patch_files` — how many files changed?
- `patch_hunks` — how many separate change blocks?
- `patch_chars` — how large is the patch as text?

Plain English:

> Is this a small, focused change or a large, spread-out change?

These are safe starting features because they can be computed directly from the candidate diff.

### B. Issue size and wording

- `problem_chars`
- `problem_lines`
- optional later: issue token count
- optional later: TF-IDF/embedding representation

Plain English:

> How much information is in the issue, and what kind of issue is it?

### C. Patch/code health checks

These are highly valuable when available:

- `patch_apply_ok`
- `syntax_ok`
- `compile_ok`
- `lint_warning_count`
- `static_error_count`
- `undefined_name_count`
- `import_error_count`

Plain English:

> Does the changed code still look structurally healthy?

### D. Test/CI evidence available before the decision

Examples:

- `visible_test_pass_rate`
- `visible_test_fail_count`
- `ci_job_failure_count`
- `ci_job_success_rate`

Plain English:

> What does the testing and CI system say about the candidate so far?

These are legitimate production features when the corresponding results are genuinely available at decision time.

### E. Independent-agent consistency

If several independent coding attempts are generated for the same issue:

- patch similarity
- same files touched
- same key symbols changed
- agreement on the affected area

Plain English:

> Do different attempts independently arrive at similar changes?

This is much stronger than assigning a confidence number by hand.

### F. Real model confidence

You may use model-generated confidence **only when it is the actual confidence output of the model**, not a number assigned after we know whether the patch succeeded.

Plain English:

> How sure was the generating system before seeing the answer?

### Features that are useful for EDA but should NOT be predictors

Never train TrustGate on:

- the gold/reference patch
- gold touched files
- `FAIL_TO_PASS`
- `PASS_TO_PASS`
- hidden SWE-bench test outcomes
- `resolved_status`
- manually assigned confidence based on success/failure
- any information created after the TrustGate decision

Those fields contain the answer or information too close to the answer.

In [ ]:
# Candidate-level dataset schema.
#
# One row = one candidate patch for one SWE-bench issue.
#
# This file does not exist until candidate patches have been generated
# and evaluated with the official SWE-bench harness.

CANDIDATE_COLUMNS = [
    "instance_id",
    "candidate_id",
    "agent_name",
    "model_name",
    "temperature",
    "patch",
    "patch_apply_ok",
    "syntax_ok",
    "compile_ok",
    "lint_warning_count",
    "static_error_count",
    "undefined_name_count",
    "import_error_count",
    "visible_test_pass_rate",
    "visible_test_fail_count",
    "ci_job_failure_count",
    "ci_job_success_rate",
    "self_consistency",
    "model_confidence",
    "resolved_status",          # FULL / PARTIAL / NO
    "fail_to_pass_rate",       # evaluation/diagnostic field
    "pass_to_pass_rate",       # evaluation/diagnostic field
    "evaluation_error",
    "label",                   # 1 only for FULL, 0 for PARTIAL/NO
]

candidate_template = pd.DataFrame(columns=CANDIDATE_COLUMNS)
display(candidate_template)

## 6. Candidate labels: how the ground truth is created

For the ML dataset, generate **real candidate patches** for SWE-bench issues.

Each candidate should be evaluated with the official SWE-bench harness.

The recommended target is:

```text
label = 1  -> FULL
label = 0  -> PARTIAL or NO
```

Keep `FULL / PARTIAL / NO` as the detailed status.

### Critical rule

An environment/evaluation failure is **not automatically label 0**.

For example:

```text
Patch is valid
      +
Docker image fails to build
      =
environment failure
```

That is different from:

```text
Patch applies
      +
tests genuinely fail
      =
candidate failure
```

The data pipeline should keep those cases separate.

### Official evaluation

The current SWE-bench tooling supports evaluating the verified benchmark and also verifying gold patches.

Examples from the current SWE-bench documentation:

```bash
swebench eval verified --gold -i <instance_id> --run-id validate-gold

swebench eval verified -p <predictions.jsonl> --run-id <run_id> -j 8
```

The official harness uses Docker-based task environments and can also run through Modal.

Source:
https://github.com/SWE-bench/SWE-bench/blob/main/docs/reference/harness.md

Source:
https://github.com/SWE-bench/SWE-bench/blob/main/docs/guides/quickstart.md

In [ ]:
# Example candidate prediction format expected by the SWE-bench harness.
example_prediction = {
    "instance_id": "owner__repo-1234",
    "model_patch": "diff --git a/example.py b/example.py\n...",
    "model_name_or_path": "TrustGate-Agent-v1",
}

print(json.dumps(example_prediction, indent=2))

## 7. Docker: where it belongs

Docker is important for **label generation**, not for pandas/scikit-learn.

Use this separation:

```text
                SWE-bench issue
                       |
             generate candidate patch
                       |
                       v
          Docker / SWE-bench harness
                       |
             execute + grade patch
                       |
                       v
                 real label
                       |
                       v
             candidate feature table
                       |
                       v
                  Colab/Python
             EDA + ML + evaluation
```

### Why Docker?

SWE-bench tasks come from different repositories and historical environments.

The official harness uses Docker to create reproducible task environments, isolate dependencies, apply the candidate patch, run tests, and grade the result.

Current SWE-bench documentation recommends roughly **120 GB free storage** and **16 GB+ RAM** for local evaluation.

For larger runs, Modal is an alternative supported by the harness.

### Recommendation

- Use **Docker or Modal** to create trustworthy candidate labels.
- Use **Colab/Jupyter/Python** for EDA, feature engineering, modeling, calibration, and evaluation.

## 8. Candidate-level EDA

Once the candidate table is populated, the most useful EDA asks:

> What characteristics are associated with successful vs unsuccessful candidate patches?

The same issue can have multiple candidate rows, so the plots below are candidate-level plots but the statistical split must still be issue-level.

In [ ]:
def add_diff_features(candidates):
    if candidates.empty:
        return candidates.copy()

    feats = pd.DataFrame(
        [unified_diff_stats(p) for p in candidates["patch"].fillna("")]
    )

    out = pd.concat(
        [
            candidates.reset_index(drop=True),
            feats.reset_index(drop=True)
        ],
        axis=1,
    )

    return out

CANDIDATE_RESULTS_PATH = None
# Example:
# CANDIDATE_RESULTS_PATH = "/content/trustgate_candidate_results.csv"

if CANDIDATE_RESULTS_PATH and os.path.exists(CANDIDATE_RESULTS_PATH):
    candidates = pd.read_csv(CANDIDATE_RESULTS_PATH)
else:
    candidates = pd.DataFrame(columns=CANDIDATE_COLUMNS)

print("Candidate rows loaded:", len(candidates))

if len(candidates):
    cand = add_diff_features(candidates)

    print("Unique issues:", cand["instance_id"].nunique())
    print("Label distribution:")
    display(cand["label"].value_counts(dropna=False).to_frame("count"))
else:
    cand = candidates.copy()
    print("No candidate results loaded yet.")

### Candidate label balance

For a real candidate dataset, check class balance before interpreting accuracy. A highly imbalanced target can make accuracy look good while the model is not useful.

In [ ]:
if len(cand) and "label" in cand.columns and cand["label"].notna().all():
    counts = cand["label"].value_counts().sort_index()
    plt.figure(figsize=(6,4))
    plt.bar(["Failure (0)", "Success (1)"], [counts.get(0,0), counts.get(1,0)])
    plt.ylabel("Candidate count")
    plt.title("Candidate label balance")
    plt.tight_layout()
    plt.show()

In [ ]:
# Candidate EDA: success rate by patch-size bucket
if len(cand) and cand["label"].notna().all():
    cand["patch_lines_changed"] = (
        cand["patch_additions"] + cand["patch_deletions"]
    )

    cand["patch_size_bucket"] = pd.qcut(
        cand["patch_lines_changed"].rank(method="first"),
        q=5,
        duplicates="drop"
    )

    success_by_bucket = cand.groupby("patch_size_bucket", observed=True)["label"].mean()

    plt.figure(figsize=(9, 4))
    plt.bar(
        [str(x) for x in success_by_bucket.index],
        success_by_bucket.values
    )
    plt.title("Candidate success rate by patch-size quintile")
    plt.ylabel("Success rate")
    plt.xlabel("Patch-size quintile")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Load real candidate results to produce candidate-level EDA.")

In [ ]:
# Candidate EDA: heatmap of numeric TrustGate features
candidate_heat_cols = [
    "patch_additions",
    "patch_deletions",
    "patch_files",
    "patch_hunks",
    "patch_chars",
    "syntax_ok",
    "compile_ok",
    "lint_warning_count",
    "static_error_count",
    "undefined_name_count",
    "import_error_count",
    "visible_test_pass_rate",
    "visible_test_fail_count",
    "ci_job_failure_count",
    "ci_job_success_rate",
    "self_consistency",
    "model_confidence",
    "label",
]

available = [c for c in candidate_heat_cols if c in cand.columns]

if len(cand) and len(available) >= 3:
    numeric = cand[available].apply(pd.to_numeric, errors="coerce")
    corr = numeric.corr()

    plt.figure(figsize=(12, 10))
    im = plt.imshow(corr.values, vmin=-1, vmax=1, aspect="auto")
    plt.colorbar(im, label="Correlation")
    plt.xticks(range(len(available)), available, rotation=70, ha="right")
    plt.yticks(range(len(available)), available)

    for i in range(len(available)):
        for j in range(len(available)):
            value = corr.iloc[i, j]
            if pd.notna(value):
                plt.text(
                    j, i, f"{value:.2f}",
                    ha="center", va="center", fontsize=7
                )

    plt.title("TrustGate candidate-feature correlation heatmap")
    plt.tight_layout()
    plt.show()
else:
    print("Load real candidate results with numeric features for the candidate heatmap.")

## 9. Train/validation/test splitting

The independent unit is the **issue**, not the candidate row.

Example:

```text
Issue 123
  candidate A
  candidate B
  candidate C
```

All three candidates must remain in the same split.

Otherwise the model can see one candidate from Issue 123 during training and another candidate from the same issue during testing.

### Recommended split

- **60%** of issues → training
- **20%** of issues → validation
- **20%** of issues → frozen test

For an even stronger experiment, also run a **repository-level holdout** where entire repositories are kept out of training.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

def grouped_train_val_test_split(
    data,
    group_col="instance_id",
    train_size=0.60,
    val_size=0.20,
    random_state=42,
):
    data = data.copy()

    gss1 = GroupShuffleSplit(
        n_splits=1,
        train_size=train_size,
        random_state=random_state,
    )

    train_idx, temp_idx = next(
        gss1.split(data, groups=data[group_col])
    )

    train = data.iloc[train_idx].copy()
    temp = data.iloc[temp_idx].copy()

    val_fraction_of_temp = val_size / (1 - train_size)

    gss2 = GroupShuffleSplit(
        n_splits=1,
        train_size=val_fraction_of_temp,
        random_state=random_state + 1,
    )

    val_idx, test_idx = next(
        gss2.split(temp, groups=temp[group_col])
    )

    val = temp.iloc[val_idx].copy()
    test = temp.iloc[test_idx].copy()

    return train, val, test

if (
    len(cand) >= 100
    and cand["label"].notna().all()
    and cand["label"].nunique() == 2
    and cand["instance_id"].nunique() >= 50
):
    train, val, test = grouped_train_val_test_split(cand)

    print("TRAIN")
    print(" rows:", len(train), "issues:", train["instance_id"].nunique())
    print("VALIDATION")
    print(" rows:", len(val), "issues:", val["instance_id"].nunique())
    print("TEST")
    print(" rows:", len(test), "issues:", test["instance_id"].nunique())
else:
    print("Need at least 100 valid candidate rows from 50+ issues before training.")

## 10. Baseline model

Start with a simple, interpretable baseline.

The first model below uses only basic patch/runtime features.

This is intentional.

We want to answer:

> “Do these signals contain predictive information at all?”

before adding embeddings, sophisticated static analysis, or large language models.

The model can later be expanded in feature families and compared using an ablation study.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

BASE_FEATURES = [
    "patch_additions",
    "patch_deletions",
    "patch_files",
    "patch_hunks",
    "patch_chars",
]

OPTIONAL_RUNTIME_FEATURES = [
    "syntax_ok",
    "compile_ok",
    "lint_warning_count",
    "static_error_count",
    "undefined_name_count",
    "import_error_count",
    "visible_test_pass_rate",
    "visible_test_fail_count",
    "ci_job_failure_count",
    "ci_job_success_rate",
    "self_consistency",
    "model_confidence",
]

def build_logistic_baseline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ])

def build_tree_baseline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            HistGradientBoostingClassifier(
                max_depth=3,
                learning_rate=0.05,
                max_iter=150,
                random_state=42,
            ),
        ),
    ])

## 11. Evaluation metrics

TrustGate needs more than one score.

### Accuracy

> Of all decisions, how many were correct?

Useful, but can be misleading with imbalanced outcomes.

### Precision

> When the model says “success,” how often is it really successful?

This is very important for an **auto-approval** gate.

### Recall

> Of all genuinely successful patches, how many did the model identify?

### F1

Balances precision and recall.

### ROC-AUC

Measures how well the model ranks successful candidates above unsuccessful ones across thresholds.

### PR-AUC

Often more informative than ROC-AUC when the positive class is uncommon.

### Specificity

> How well does the model avoid false positives?

### Brier score

Measures the quality of predicted probabilities.

### Calibration / ECE

A model that says `0.90` should be correct roughly 90% of the time in that probability range.

### TrustGate operational metrics

We also need:

- auto-approve precision
- auto-reject precision
- human-review rate
- automatic-decision coverage
- false-approval rate
- false-rejection rate

For TrustGate, these operational measures may matter more than raw accuracy.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)

def classification_metrics(y_true, p, threshold=0.5):
    y_true = np.asarray(y_true)
    p = np.asarray(p)
    y_pred = (p >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    result = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "specificity": tn / max(1, tn + fp),
        "brier_score": brier_score_loss(y_true, p),
    }

    if len(np.unique(y_true)) == 2:
        result["roc_auc"] = roc_auc_score(y_true, p)
        result["pr_auc"] = average_precision_score(y_true, p)
    else:
        result["roc_auc"] = np.nan
        result["pr_auc"] = np.nan

    return result

def fit_model_and_predict(train, test, features, builder):
    model = builder()

    model.fit(
        train[features],
        train["label"].astype(int)
    )

    p = model.predict_proba(test[features])[:, 1]

    metrics = classification_metrics(
        test["label"].astype(int).values,
        p
    )

    return model, p, metrics

In [ ]:
# Train the simple baseline when enough real candidate data exists.

if (
    len(cand) >= 100
    and cand["label"].notna().all()
    and cand["label"].nunique() == 2
    and cand["instance_id"].nunique() >= 50
):
    train, val, test = grouped_train_val_test_split(cand)

    model, p_test, baseline_metrics = fit_model_and_predict(
        train,
        test,
        BASE_FEATURES,
        build_logistic_baseline,
    )

    display(pd.Series(baseline_metrics, name="value").to_frame())

else:
    print(
        "Metrics intentionally not produced yet because the candidate dataset "
        "is not sufficiently populated."
    )

## 12. ROC and Precision-Recall curves

These curves show performance across many decision thresholds.

- ROC curve: true-positive rate vs false-positive rate.
- Precision-recall curve: precision vs recall.

The important point is that TrustGate should not pick a threshold because it looks nice on a graph. Thresholds should be selected on validation data using the business consequences of errors.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

if (
    "test" in globals()
    and len(test)
    and len(np.unique(test["label"])) == 2
):
    y_test = test["label"].astype(int).values

    fpr, tpr, _ = roc_curve(y_test, p_test)
    precision, recall, _ = precision_recall_curve(y_test, p_test)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr)
    plt.plot([0, 1], [0, 1], "--")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title("ROC curve")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall curve")
    plt.tight_layout()
    plt.show()
else:
    print("ROC/PR plots require a real test split containing both classes.")

## 13. Calibration

TrustGate produces a probability, not just a class.

A probability is useful only if it is reasonably calibrated.

Example:

```text
Model says 0.90
     |
     v
Candidates in this range
should succeed about 90% of the time
```

A reliability diagram compares:

- predicted probability
- observed success rate

The notebook also calculates Expected Calibration Error (ECE).

In [ ]:
def expected_calibration_error(y_true, p, bins=10):
    y_true = np.asarray(y_true)
    p = np.asarray(p)

    edges = np.linspace(0, 1, bins + 1)
    ece = 0.0

    for i in range(bins):
        lo, hi = edges[i], edges[i + 1]

        if i == bins - 1:
            mask = (p >= lo) & (p <= hi)
        else:
            mask = (p >= lo) & (p < hi)

        if mask.any():
            ece += (
                mask.mean()
                * abs(y_true[mask].mean() - p[mask].mean())
            )

    return ece

def reliability_points(y_true, p, bins=10):
    y_true = np.asarray(y_true)
    p = np.asarray(p)

    edges = np.linspace(0, 1, bins + 1)
    predicted = []
    observed = []

    for i in range(bins):
        lo, hi = edges[i], edges[i + 1]

        if i == bins - 1:
            mask = (p >= lo) & (p <= hi)
        else:
            mask = (p >= lo) & (p < hi)

        if mask.any():
            predicted.append(p[mask].mean())
            observed.append(y_true[mask].mean())

    return np.asarray(predicted), np.asarray(observed)

if "test" in globals() and len(test):
    pred_prob, observed_rate = reliability_points(
        test["label"].astype(int).values,
        p_test,
    )

    plt.figure(figsize=(6, 5))
    plt.plot([0, 1], [0, 1], "--")
    plt.plot(pred_prob, observed_rate, "o-")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed success rate")
    plt.title("TrustGate reliability diagram")
    plt.tight_layout()
    plt.show()

    print(
        "ECE:",
        expected_calibration_error(
            test["label"].astype(int).values,
            p_test
        )
    )

## 14. The three-way TrustGate decision

The model produces:

```text
P(candidate is correct)
```

Then TrustGate creates a policy:

```text
             probability
                  |
      +-----------+-----------+
      |                       |
   high                    low
      |                       |
  APPROVE                  REJECT

        middle range
              |
        HUMAN REVIEW
```

Example thresholds:

- `p >= 0.85` → approve
- `p <= 0.20` → reject
- otherwise → human review

These are **illustrative only**.

The actual thresholds must be chosen on the validation set and then frozen before testing.

That is where the business risk enters the ML system:

- false approval can be expensive
- false rejection can slow delivery
- human review has an operational cost

The model should not choose those business trade-offs by itself.

In [ ]:
def apply_three_way_policy(
    p,
    approve_threshold=0.85,
    reject_threshold=0.20,
):
    p = np.asarray(p)

    return np.where(
        p >= approve_threshold,
        "approve",
        np.where(
            p <= reject_threshold,
            "reject",
            "human_review"
        )
    )

def gate_metrics(
    y_true,
    p,
    approve_threshold=0.85,
    reject_threshold=0.20,
):
    y_true = np.asarray(y_true)
    decisions = apply_three_way_policy(
        p,
        approve_threshold,
        reject_threshold
    )

    approved = decisions == "approve"
    rejected = decisions == "reject"
    reviewed = decisions == "human_review"

    return {
        "auto_approve_rate": approved.mean(),
        "auto_reject_rate": rejected.mean(),
        "human_review_rate": reviewed.mean(),
        "automatic_decision_coverage": (~reviewed).mean(),

        "approve_precision": (
            y_true[approved].mean()
            if approved.any()
            else np.nan
        ),

        "reject_precision": (
            (1 - y_true[rejected]).mean()
            if rejected.any()
            else np.nan
        ),

        "false_approval_rate": (
            (1 - y_true[approved]).mean()
            if approved.any()
            else np.nan
        ),

        "false_rejection_rate": (
            y_true[rejected].mean()
            if rejected.any()
            else np.nan
        ),
    }

print(
    "Tune thresholds on validation data, then freeze them before the final test."
)

In [ ]:
# Validation threshold grid.
#
# We do not select thresholds on the test set.

def threshold_grid(val_df, model, features):
    p_val = model.predict_proba(val_df[features])[:, 1]
    y_val = val_df["label"].astype(int).values

    rows = []

    for approve_t in np.arange(0.60, 0.96, 0.05):
        for reject_t in np.arange(0.05, 0.41, 0.05):

            if reject_t >= approve_t:
                continue

            metrics = gate_metrics(
                y_val,
                p_val,
                approve_threshold=approve_t,
                reject_threshold=reject_t,
            )

            metrics["approve_threshold"] = approve_t
            metrics["reject_threshold"] = reject_t

            rows.append(metrics)

    return pd.DataFrame(rows)

if "val" in globals() and len(val):
    grid = threshold_grid(
        val,
        model,
        BASE_FEATURES
    )

    display(grid.head())
else:
    print("Validation data are required for threshold selection.")

## 15. Confidence intervals

Candidate patches are correlated because several candidates can belong to the same issue.

Therefore, ordinary row-level confidence intervals can look more certain than they should.

For final reporting, bootstrap at the **issue level**:

1. sample `instance_id` values with replacement
2. include all candidate rows belonging to each sampled issue
3. recompute the metric
4. report the 2.5th and 97.5th percentiles

This treats the issue as the independent unit.

In [ ]:
from sklearn.metrics import roc_auc_score

def grouped_bootstrap_auc(
    data,
    group_col="instance_id",
    label_col="label",
    probability_col="predicted_probability",
    n_boot=1000,
    random_state=42,
):
    rng = np.random.default_rng(random_state)
    groups = data[group_col].dropna().unique()

    grouped = {
        g: data[data[group_col] == g]
        for g in groups
    }

    values = []

    for _ in range(n_boot):
        sampled_groups = rng.choice(
            groups,
            size=len(groups),
            replace=True
        )

        sample = pd.concat(
            [grouped[g] for g in sampled_groups],
            ignore_index=True
        )

        y = sample[label_col].astype(int).values
        p = sample[probability_col].astype(float).values

        if len(np.unique(y)) < 2:
            continue

        values.append(roc_auc_score(y, p))

    if not values:
        return {
            "mean_auc": np.nan,
            "lower_95": np.nan,
            "upper_95": np.nan,
        }

    values = np.asarray(values)

    return {
        "mean_auc": float(values.mean()),
        "lower_95": float(np.quantile(values, 0.025)),
        "upper_95": float(np.quantile(values, 0.975)),
    }

## 16. Recommended dataset size

For a first serious ML study, a practical target is:

**200–500 distinct SWE-bench issues × 5–10 candidate patches per issue**

That produces roughly:

**1,000–5,000 candidate-patch rows**

The important thing is not just more rows. It is **diverse, independently generated candidate patches with real evaluation labels**.

Prefer:

- multiple independent attempts per issue
- more than one coding model when possible
- all 12 SWE-bench Verified repositories
- multiple difficulty levels
- successful and unsuccessful candidates
- genuine patch diversity

Synthetic mutations can be useful as smoke-test data, but should not be the main evidence for ML performance.

### Production data comes later

SWE-bench is an excellent benchmark layer.

For a production TrustGate system, eventually add historical PR data containing things such as:

- visible CI results
- human review outcome
- merge/reject outcome
- post-merge rollback/revert signals
- security/static-analysis findings
- repository and language context

That would let you move from:

> “Does the patch solve a benchmark issue?”

to:

> “Is this real PR safe and appropriate for automated handling?”

## 17. Recommended feature roadmap

Build the model in stages.

### Stage 1 — Patch shape

Start with:

- additions
- deletions
- files changed
- hunks
- patch size

Question answered:

> Can simple patch geometry tell us anything?

### Stage 2 — Code health

Add:

- patch applies
- syntax/compile checks
- lint warnings
- static-analysis errors
- import/undefined-name checks

Question:

> Does the patch remain technically healthy?

### Stage 3 — Tests and CI

Add only results genuinely available at decision time:

- visible test pass rate
- CI failures
- CI success rate

Question:

> What evidence does the engineering pipeline provide?

### Stage 4 — Agreement

Add:

- self-consistency
- patch similarity
- changed-file agreement

Question:

> Do independent attempts agree?

### Stage 5 — Semantic features

Only after the above:

- TF-IDF
- code embeddings
- issue/patch similarity
- model-derived representations

Question:

> Does understanding the meaning of the issue and change add predictive value?

### Stage 6 — Production TrustGate labels

Replace or supplement SWE-bench labels with real organizational outcomes.

This is where TrustGate becomes a production decision system rather than a benchmark classifier.

## 18. Final architecture

```text
               SWE-bench Verified
                500 real issues
                       |
        +--------------+--------------+
        |                             |
   understand task              generate candidates
        |                             |
        v                             v
       EDA                    Docker / Modal evaluation
                                      |
                                      v
                                candidate labels
                                      |
                                      v
                         TrustGate feature engineering
                                      |
                                      v
                                  ML model
                                      |
                         calibrated probability
                                      |
             +------------------------+------------------------+
             |                        |                        |
        high probability       middle probability        low probability
             |                        |                        |
          APPROVE               HUMAN REVIEW                REJECT
```

### What this notebook gives you

- benchmark-level EDA
- repository and difficulty analysis
- patch-size distributions
- test/issue complexity analysis
- correlation heatmap
- missing-value heatmap
- candidate-level EDA
- candidate-feature heatmap
- explicit feature definitions
- label schema
- Docker/Modal evaluation workflow
- issue-level train/validation/test splitting
- baseline logistic model
- ROC-AUC
- PR-AUC
- accuracy
- precision
- recall
- F1
- balanced accuracy
- specificity
- Brier score
- calibration / ECE
- reliability diagram
- three-way TrustGate policy
- threshold tuning
- issue-level bootstrap confidence intervals

### Bottom line

SWE-bench Verified gives TrustGate **real software issues and an objective way to test candidate patches**.

It does not directly give us a ready-made “approve / human-review / reject” training set.

The ML dataset we build is:

```text
candidate PR patch
        +
issue/code context
        +
runtime/CI/static signals
        |
        v
SWE-bench evaluation
        |
        v
FULL / PARTIAL / NO
        |
        v
ML label
```

The strongest first model should be simple, leakage-safe, calibrated, and evaluated on issues that were never used for training.

## References

1. SWE-bench Verified — Hugging Face dataset card  
   https://huggingface.co/datasets/princeton-nlp/SWE-bench_Verified

2. SWE-bench evaluation tutorial  
   https://github.com/SWE-bench/SWE-bench/blob/main/docs/assets/evaluation.md

3. SWE-bench evaluation harness reference  
   https://github.com/SWE-bench/SWE-bench/blob/main/docs/reference/harness.md

4. SWE-bench quick start  
   https://github.com/SWE-bench/SWE-bench/blob/main/docs/guides/quickstart.md

5. SWE-bench Docker setup  
   https://github.com/SWE-bench/SWE-bench/blob/main/docs/guides/docker_setup.md